<a href="https://colab.research.google.com/github/Dyuko/DataScience/blob/main/rag_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Alumno: Matías Irala

CI: 4637852

# Parte 1: Preparación del entorno

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-core
!pip install langchain-groq

In [ ]:
!pip install faiss-cpu

In [ ]:
import os

import dotenv
from google.colab import drive

drive.mount('/content/drive')

dotenv.load_dotenv('/content/drive/MyDrive/.env')

api_key = os.environ.get('GROP_API_KEY')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Parte 2: Construcción del sistema RAG

## 1. Carga de documentos

In [ ]:
# Cargar csv EdSheeran.csv desde drive
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/datos/EdSheeran.csv').dropna(subset=["Lyric"])

In [ ]:
#df.head()

In [ ]:
from langchain_core.documents import Document
# Convertir cada fila en un Document de LangChain
pages = []
for _, row in df.iterrows():
    # El contenido principal será la letra (Lyric)
    content = row["Lyric"]

    # Metadatos (opcional, pero útil para filtrado)
    metadata = {
        "id": row["Id"],
        "artist": row["Artist"],
        "title": row["Title"],
        "album": row["Album"],
        "year": row["Year"],
        "date": row["Date"],
    }

    pages.append(Document(page_content=content, metadata=metadata))

In [ ]:
#pages

## 2. División en fragmentos

Usa CharacterTextSplitter para dividir los documentos en chunks de 500 caracteres.

In [ ]:
import langchain
langchain.verbose = True

In [ ]:
## Split and Store
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Initialize splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

## Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

## Split documents
documents = text_splitter.split_documents(pages)

In [ ]:
#documents

## 3. Creación del índice semántico

Usa FAISS para construir un índice vectorial a partir de los chunks usando embeddings como OpenAIEmbeddings.

In [ ]:
## Vectorize documents into vector store
from langchain_community.vectorstores import FAISS
vector = FAISS.from_documents(documents,  ## documents
                              embeddings) ## embedding model

## 4. Configuración del LLM

Usa OpenAIChat o ChatOpenAI si estás usando GPT-3.5 o (LLamaCpp o ollama) si estás trabajando localmente.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

chat = ChatGroq(temperature=0.3, groq_api_key=api_key, model_name="llama3-70b-8192")

## 5. 5. Construcción del RAG chain

Crea un RetrievalQA chain que combine el retriever (índice FAISS) con el modelo generador (LLM).

In [ ]:
# Define prompt template for RAG
from langchain.chains import RetrievalQA
from langchain_core.prompts import ChatPromptTemplate

## Define prompt template
template = """You are an expert at analyzing song lyrics. Answer the user's question
using ONLY the following lyric excerpts. Do not use external knowledge.

If the excerpts don't contain enough information to answer,
say: "I can't find relevant lyrics to answer this."

Context (lyric excerpts):
{context}

Question: {question}

Answer (based ONLY on the above lyrics, always including the song name when mentioning lyrics):"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=chat,
    chain_type="stuff",
    retriever=vector.as_retriever(search_kwargs={"k": 5}),  # Reduced to 5 for more concise answers
    chain_type_kwargs={
        "prompt": prompt,
        "document_prompt": ChatPromptTemplate.from_template(
            "Song: {title}\nAlbum: {album}\nYear: {year}\nExcerpt: {page_content}"
        )
    },
    return_source_documents=True
)

## 6. Evaluación del sistema

Formula al menos 5 preguntas que solo puedan responderse con los documentos cargados.

Evalúa la calidad y precisión de las respuestas.

In [ ]:
def ask_question(question):
    print(f"\n\033[1mPregunta:\033[0m {question}")
    result = qa_chain.invoke({"query": question})

    print("\n\033[1mRespuesta:\033[0m")
    print(result["result"])

In [ ]:
ask_question("Which Ed Sheeran songs are about love?")


Pregunta: Which Ed Sheeran songs are about love?


> Entering new RetrievalQA chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Human: You are an expert at analyzing song lyrics. Answer the user's question
using ONLY the following lyric excerpts. Do not use external knowledge.

If the excerpts don't contain enough information to answer,
say: "I can't find relevant lyrics to answer this."

Context (lyric excerpts):
Human: Song: Summer Nights/No Love For The Lonely (Ed Demo)
Album: nan
Year: nan
Excerpt: ed sheeran no lololove for the lonely no lololove for the lonely no lololove for the lonely no lololove for the lonely   ed sheeran another night another club another room a new one when morning comes i'm running off i found lust but i wanna find love and every time i get close i run wild and explode but i'm lonely in la  pre ed sheeran no lie yeah my mama never raised me to be a bad boy to chase women and to only break he

In [ ]:
ask_question("Can you name any lyrics that talk about nostalgia?")


Pregunta: Can you name any lyrics that talk about nostalgia?


> Entering new RetrievalQA chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Human: You are an expert at analyzing song lyrics. Answer the user's question
using ONLY the following lyric excerpts. Do not use external knowledge.

If the excerpts don't contain enough information to answer,
say: "I can't find relevant lyrics to answer this."

Context (lyric excerpts):
Human: Song: Oh No
Album: nan
Year: nan
Excerpt: think it's written on a notebook somewhere can you feel it as you breathe the night air yeah i can't remember what you said to me but i know it was hard hard hard for you to speak  time drift as you speed into love heart race as you heat up my blood yeaaah let's just continue what we used to do cause i know it was hard hard hard to see the truth  and it's late it's late  i can't let go oh no i miss you oh no i can't let go oh no i miss you oh no  oooo

In [ ]:
ask_question("What song mentions cigarette?")


Pregunta: What song mentions cigarette?


> Entering new RetrievalQA chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Human: You are an expert at analyzing song lyrics. Answer the user's question
using ONLY the following lyric excerpts. Do not use external knowledge.

If the excerpts don't contain enough information to answer,
say: "I can't find relevant lyrics to answer this."

Context (lyric excerpts):
Human: Song: Wake Me Up
Album: + (Plus)
Year: 2011.0
Excerpt: of smoke you always try and get me to stop but you drink as much as me and i get drunk a lot so ill take you to the beach and walk along the sand and i'll make you a heart pendant with a pebble held in my hand and i'll carve it like this necklace so the heart falls where your chest is now a piece of me is a piece of the beach and it falls just where it needs to be and rests peacefully so you just need to breathe to feel my heart against yours now against your

In [ ]:
ask_question("What song mentions arise from my tomb ?")


Pregunta: What song mentions arise from my tomb ?


> Entering new RetrievalQA chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Human: You are an expert at analyzing song lyrics. Answer the user's question
using ONLY the following lyric excerpts. Do not use external knowledge.

If the excerpts don't contain enough information to answer,
say: "I can't find relevant lyrics to answer this."

Context (lyric excerpts):
Human: Song: Small Bump - Live From Wembley Stadium
Album: nan
Year: nan
Excerpt: spoken  now wembley i've not played this song in a around two years maybe maybe longer but i feel like playing it today let's see how it goes   you're just a small bump unborn in four months you're brought to life you might be left with my hair but you'll have your mother's eyes i'll hold your body in my hands be as gentle as i can but for now you're a scan of my unmade plans a small bump in four months you're brought to life  pr

In [ ]:
ask_question("Can you name any lyrics that talk about Nightmares?")


Pregunta: Can you name any lyrics that talk about Nightmares?


> Entering new RetrievalQA chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Human: You are an expert at analyzing song lyrics. Answer the user's question
using ONLY the following lyric excerpts. Do not use external knowledge.

If the excerpts don't contain enough information to answer,
say: "I can't find relevant lyrics to answer this."

Context (lyric excerpts):
Human: Song: Nightmares - + Random Impulse + Sway + Wretch 32
Album: nan
Year: nan
Excerpt: in i feel my nightmares watching me and when my dreams are sleeping i feel my nightmares watching me oh oh oh i feel my nightmares watching me they watching me they watch me sleep  let me go i fell asleep on sofa walk up in reality and daydream about losing my sanity i've been rhyming forever got a blind flow so i can see an off note with my eyes closed but to achieve the dreams you can't doze off cos your d

Calidad y precisión de las respuestas:

* Las respuestas generadas muestran alta precisión cuando el contexto relevante es recuperado por el sistema de retrieval.
* Se observa coherencia semántica entre las preguntas formuladas y los fragmentos de letras proporcionados como evidencia.
* El principal desafío identificado radica en la efectividad del componente de retrieval. En el caso de la pregunta "What song mentions cigarette?", aunque la canción "Castle On The Hill (Acoustic)" contiene explícitamente la palabra "cigarette" en su letra, el sistema no la incluyó en el contexto proporcionado al LLM.